In [ ]:
import pandas as pd
import numpy as np
import torch
import os
import sys
from tqdm import tqdm, trange
os.chdir("src")

# sys.path.append("../../")
import biked_commons
from biked_commons.design_evaluation.design_evaluation import *
from biked_commons.resource_utils import split_datasets_path
from biked_commons.conditioning import conditioning
# from biked_commons.design_evaluation.scoring import *

c:\Users\Lyler\mambaforge\envs\torch\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\Lyler\mambaforge\envs\torch\Lib\site-packages\sklearn\base.py:380: InconsistentVersionWarning: Trying to unpickle estimator MinMaxScaler from version 1.6.1 when using version 1.6.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
c:\Users\Lyler\Documents\biked-commons\src\biked_commons\design_evaluation\../..\biked_commons\prediction\usability_predictors.py:37: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data whic

In [3]:
data = pd.read_csv(split_datasets_path("bike_bench.csv"), index_col=0)

#sample 100
data = data.sample(100, random_state=0)
data_tens = torch.tensor(data.values, dtype=torch.float32)

In [4]:
StandardEvaluations: List[EvaluationFunction] = [
    # UsabilityEvaluator(),
    AeroEvaluator(),
    ]
# evaluator = construct_dataframe_evaluator(StandardEvaluations)
evaluator, requirement_names, requirement_types = construct_tensor_evaluator(StandardEvaluations, data.columns)
# isobjective = torch.tensor(requirement_types) == 1


In [5]:
num_data = data.shape[0]
rider_condition = conditioning.sample_riders(num_data, split="test")
use_case_condition = conditioning.sample_use_case(num_data, split="test")
text_condition = conditioning.sample_text(num_data, split="test")
image_embeddings = conditioning.sample_image_embedding(num_data, split="test")
condition = {"Rider": rider_condition, "Use Case": use_case_condition, "Embedding": image_embeddings}
# condition = {"Rider": rider_condition, "Use Case": use_case_condition, "Text": text_condition}

In [6]:
eval_scores = evaluator(data_tens, condition)
# eval_scores = evaluator(data, condition)
eval_scores

c:\Users\Lyler\mambaforge\envs\torch\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(
c:\Users\Lyler\Documents\biked-commons\src\biked_commons\design_evaluation\../..\biked_commons\prediction\aero_predictor.py:29: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  upper_leg_width = torch.tensor((torso_width/2 - 0.16)/2 + 0.14, device=device)


In [11]:
isobjective = torch.tensor(requirement_types) == 1
objective_scores = eval_scores[:, isobjective].detach().numpy()
constraint_scores = eval_scores[:, ~isobjective].detach().numpy()

In [14]:
objective_scores

array([[ 4.10594523e-01,  2.32913742e+01,  3.36137314e+01,
         7.41195602e+01,  9.16180267e+01,  5.26587963e+00,
         1.82000101e-01,  8.43692303e-01,  1.21483350e+00],
       [ 6.11982107e-01,  2.30991440e+01,  3.06382141e+01,
         7.90668335e+01,  9.24738007e+01,  4.85874653e+00,
        -1.36101794e+00,  1.40175509e+00,  1.00016665e+00],
       [ 5.19893765e-01,  2.11897202e+01,  0.00000000e+00,
         8.29077682e+01,  1.00914032e+02,  5.80406141e+00,
         6.66143715e-01,  6.78761423e-01,  1.42864692e+00],
       [ 5.27838528e-01,  1.92369499e+01,  7.27486267e+01,
         5.53944931e+01,  9.30236816e+01,  4.86958551e+00,
        -1.99992537e+00,  1.46680331e+00,  1.04408967e+00],
       [ 3.06596041e-01,  2.11568489e+01,  0.00000000e+00,
         9.49430084e+01,  1.01009979e+02,  5.06657457e+00,
         1.34696722e+00,  5.54188132e-01,  1.13879180e+00],
       [ 4.37294960e-01,  2.45644608e+01,  2.79378281e+01,
         7.38727875e+01,  1.00041763e+02,  5.846687

In [15]:
constraint_scores

array([[   0.36401975,    0.50710106, -160.        , ...,  -20.706543  ,
        -109.49753   ,   -0.5       ],
       [  -0.47948968,    0.36673903, -188.20001   , ...,  -21.5448    ,
        -164.5       ,   -0.5       ],
       [   0.5602257 ,    0.542994  , -180.        , ...,  -70.85065   ,
        -102.5       ,   -0.5       ],
       ...,
       [   1.1084493 ,    0.57821405, -100.099976  , ...,  -60.968018  ,
         -92.5       ,   -0.5       ],
       [  -0.5618546 ,    0.43819606,  -27.        , ...,  -68.14484   ,
        -164.5       ,   -0.5       ],
       [   0.4051957 ,    0.50286067, -169.        , ...,  -57.373535  ,
        -107.        ,   -0.5       ]], shape=(100, 12), dtype=float32)

In [12]:
main_scorer = construct_scorer(MainScores, StandardEvaluations, data.columns)
detailed_scorer = construct_scorer(DetailedScores, StandardEvaluations, data.columns)

Calculating reference point for scoring functions...


c:\Users\Lyler\mambaforge\envs\torch\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(
c:\Users\Lyler\Documents\biked-commons\src\biked_commons\design_evaluation\../..\biked_commons\prediction\aero_predictor.py:29: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  upper_leg_width = torch.tensor((torso_width/2 - 0.16)/2 + 0.14, device=device)


Calculating reference point for scoring functions...


c:\Users\Lyler\mambaforge\envs\torch\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(
c:\Users\Lyler\Documents\biked-commons\src\biked_commons\design_evaluation\../..\biked_commons\prediction\aero_predictor.py:29: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  upper_leg_width = torch.tensor((torso_width/2 - 0.16)/2 + 0.14, device=device)


In [13]:
main_scorer(data_tens, condition)

c:\Users\Lyler\mambaforge\envs\torch\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(
c:\Users\Lyler\Documents\biked-commons\src\biked_commons\design_evaluation\../..\biked_commons\prediction\aero_predictor.py:29: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  upper_leg_width = torch.tensor((torso_width/2 - 0.16)/2 + 0.14, device=device)


Hypervolume                     0.000000
Constraint Satisfaction Rate    0.844167
Maximum Mean Discrepancy        0.003187
dtype: float64

In [ ]:
detailed_scorer(data_tens, condition)

c:\Users\Lyler\mambaforge\envs\torch\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(
c:\Users\Lyler\Documents\biked-commons\src\biked_commons\design_evaluation\../..\biked_commons\prediction\aero_predictor.py:29: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  upper_leg_width = torch.tensor((torso_width/2 - 0.16)/2 + 0.14, device=device)


Min Objective Score: Usability Score - 0 to 1                                                                 0.791411
Min Objective Score: Drag Force                                                                              27.735741
Min Objective Score: Knee Angle Error                                                                       186.338135
Min Objective Score: Hip Angle Error                                                                        822.850403
Min Objective Score: Arm Angle Error                                                                        861.650940
Min Objective Score: Mass                                                                                    22.190498
Min Objective Score: Planar Compliance                                                                      180.347916
Min Objective Score: Transverse Compliance                                                                  265.026398
Min Objective Score: Eccentric Compliance       